# Case 3: Urumqi Residential Fire & Zero COVID Policy Reversal

**Authors:** Hanyu Xie (hx2413), Haotian Lei (hl3945)

This notebook analyzes how social media pressure following the **Urumqi residential fire (Nov 24, 2022)** triggered unprecedented public protests that forced a **government policy reversal** (end of Zero COVID, Dec 7, 2022), producing measurable gains in reopening-sensitive stocks.

Unlike Cases 1 and 2 (direct trend → market), this case documents a **three-stage chain**:
> Social Media Pressure → Government Policy Reversal → Market Recovery

**Stocks tracked:** Air China (0753.HK), Haidilao (6862.HK), Trip.com (TCOM)  
**Period:** Oct 2022 – Feb 2023

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import grangercausalitytests, adfuller
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_style('whitegrid')

import os
os.makedirs('figures', exist_ok=True)
print('All imports successful.')

## 2. Data Loading

| Source | Description |
|--------|-------------|
| [Investing.com](https://www.investing.com/) | Historical daily OHLCV stock prices |
| [Google Trends](https://trends.google.com/) | Weekly search interest index (0-100) |
| Event timeline | Key events from news and government announcements |

In [ ]:
def parse_volume(v):
    if pd.isna(v) or v == '-':
        return 0
    v = str(v).strip().replace(',', '')
    for suffix, mult in {'K': 1e3, 'M': 1e6, 'B': 1e9}.items():
        if v.endswith(suffix):
            return float(v[:-1]) * mult
    try:
        return float(v)
    except:
        return 0

def load_investing_csv(filepath, name='Stock'):
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.strip().str.replace('\ufeff', '')
    df['date'] = pd.to_datetime(df['Date'])
    df['close'] = df['Price'].astype(str).str.replace(',', '').astype(float)
    for col in ['Open', 'High', 'Low']:
        df[col.lower()] = df[col].astype(str).str.replace(',', '').astype(float)
    df['volume'] = df['Vol.'].apply(parse_volume)
    df = df[['date', 'close', 'open', 'high', 'low', 'volume']].sort_values('date').reset_index(drop=True)
    df['daily_return'] = df['close'].pct_change()
    df['cum_return'] = (1 + df['daily_return']).cumprod() - 1
    print(f"{name}: {len(df)} days, {df['date'].min().date()} to {df['date'].max().date()}, price {df['close'].min():.2f}-{df['close'].max():.2f}")
    return df

def load_gtrends_csv(filepath, name='Trends'):
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.strip().str.replace('"', '')
    date_col = df.columns[0]
    df = df.rename(columns={date_col: 'date'})
    df['date'] = pd.to_datetime(df['date'])
    for col in df.columns[1:]:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace('<1', '0'), errors='coerce')
    df = df.sort_values('date').reset_index(drop=True)
    print(f"{name}: {len(df)} weeks, {df['date'].min().date()} to {df['date'].max().date()}, cols: {list(df.columns[1:])}")
    return df

In [ ]:
airchina_stock  = load_investing_csv('data/air_china_stock.csv',  'Air China (0753.HK)')
haidilao_stock  = load_investing_csv('data/haidilao_stock.csv',   'Haidilao (6862.HK)')
tripcom_stock   = load_investing_csv('data/trip_com_stock.csv',   'Trip.com (TCOM)')
urumqi_trends   = load_gtrends_csv('data/urumqi_trends.csv',      'Urumqi Crisis Trends')

## 3. Data Alignment

Align daily stock data to weekly Google Trends frequency.

In [ ]:
def align_stock_to_weekly(stock_df, trends_df):
    stock_weekly = stock_df.set_index('date').resample('W-SUN').agg(
        {'close': 'mean', 'volume': 'sum', 'daily_return': 'sum'}).reset_index()
    stock_weekly.columns = ['date', 'avg_close', 'total_volume', 'weekly_return']
    merged = pd.merge_asof(trends_df.sort_values('date'), stock_weekly.sort_values('date'),
                           on='date', direction='nearest', tolerance=pd.Timedelta('7D'))
    merged = merged.dropna(subset=['avg_close'])
    print(f"Aligned: {len(merged)} weekly observations")
    return merged

urumqi_merged_ac = align_stock_to_weekly(airchina_stock,  urumqi_trends)
urumqi_merged_hd = align_stock_to_weekly(haidilao_stock,  urumqi_trends)
urumqi_merged_tc = align_stock_to_weekly(tripcom_stock,   urumqi_trends)

urumqi_tcols = [c for c in urumqi_merged_ac.columns
                if c not in ['date', 'avg_close', 'total_volume', 'weekly_return']]
print(f"\nTrend columns: {urumqi_tcols}")

## 4. Event Timeline

In [ ]:
urumqi_events = {
    '2022-11-24': 'Urumqi fire\n(10 dead)',
    '2022-11-27': 'White Paper protests\ngo viral',
    '2022-12-07': 'Zero COVID ended\n(新十条)',
    '2022-12-26': 'Quarantine\nrules lifted',
}

# Protest pressure window: fire to policy reversal
protest_period   = (pd.Timestamp('2022-11-24'), pd.Timestamp('2022-12-07'))
# Reopening rally window: policy reversal onwards
reopening_start  = pd.Timestamp('2022-12-07')

print('Event timeline defined.')
for d, e in urumqi_events.items():
    print(f"  {d}: {e.replace(chr(10), ' | ')}")

## 5. Visualization

### 5.1 Figure 1: Search Interest vs. Air China Stock Price

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 7))

# Shade protest pressure window
ax1.axvspan(*protest_period, alpha=0.10, color='red',  label='Protest Pressure Period')
ax1.axvspan(reopening_start, pd.Timestamp('2023-02-28'), alpha=0.06, color='green', label='Reopening Rally Period')

# Trend lines
colors_t = ['#E74C3C', '#3498DB', '#8E44AD', '#27AE60']
plot_cols = ['China protests', 'zero covid']          # two most informative
for i, col in enumerate(plot_cols):
    ax1.plot(urumqi_merged_ac['date'], urumqi_merged_ac[col], '-o',
             color=colors_t[i], markersize=4, linewidth=2,
             label=f'Google Trends: "{col}"')
ax1.set_ylabel('Google Trends Search Interest (0-100)', fontsize=12, color='#E74C3C')
ax1.set_ylim(0, 115)

# Air China stock (right axis)
ax2 = ax1.twinx()
ax2.plot(airchina_stock['date'], airchina_stock['close'], '-s',
         color='#F39C12', markersize=3, linewidth=2.5, label='Air China 0753.HK (HKD)')
ax2.set_ylabel('Air China Stock Price (HKD)', fontsize=12, color='#F39C12')

# Key finding annotation
peak_protests = urumqi_merged_ac['China protests'].max()
ac_trough = airchina_stock['close'].min()
ac_peak   = airchina_stock[airchina_stock['date'] >= reopening_start]['close'].max()
ax1.annotate(
    f'KEY FINDING:\n"China protests" peaked at {peak_protests} on week of Nov 27.\n'
    f'Government ended Zero COVID 13 days later.\n'
    f'Air China rallied from {ac_trough:.2f} \u2192 {ac_peak:.2f} HKD (+{(ac_peak/ac_trough-1)*100:.0f}%).',
    xy=(pd.Timestamp('2022-10-05'), 62), fontsize=8.5,
    bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', alpha=0.9))

# Event markers
for ds, label in urumqi_events.items():
    dt = pd.Timestamp(ds)
    ax2.axvline(x=dt, color='gray', linestyle='--', alpha=0.5)
    ax2.annotate(label, xy=(dt, ax2.get_ylim()[1] * 0.94), fontsize=7.5,
                 ha='center', va='top',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=9)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax1.xaxis.set_major_locator(mdates.MonthLocator())
ax1.set_xlabel('Date')
plt.title('Social Media Search Interest vs. Air China Stock Price\n'
          'Urumqi Fire & Zero COVID Policy Reversal (Oct 2022 – Feb 2023)',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/C3_Fig1_Urumqi_Trends_vs_AirChina.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Figure 2: Air China Stock Price & Volume

In [ ]:
fig, (ax_p, ax_v) = plt.subplots(2, 1, figsize=(14, 9),
                                  height_ratios=[3, 1], sharex=True,
                                  gridspec_kw={'hspace': 0.05})
for ax in [ax_p, ax_v]:
    ax.axvspan(*protest_period, alpha=0.10, color='red')
    ax.axvspan(reopening_start, pd.Timestamp('2023-02-28'), alpha=0.06, color='green')

ax_p.plot(airchina_stock['date'], airchina_stock['close'],
          color='#2C3E50', linewidth=1.5, label='Air China Closing Price (HKD)')

mi = airchina_stock['close'].idxmin()
mx_post = airchina_stock[airchina_stock['date'] >= reopening_start]['close'].idxmax()
ax_p.annotate(f"Trough: {airchina_stock.loc[mi,'close']:.2f} HKD",
              xy=(airchina_stock.loc[mi,'date'], airchina_stock.loc[mi,'close']),
              xytext=(15, 15), textcoords='offset points', fontsize=9,
              color='red', fontweight='bold',
              arrowprops=dict(arrowstyle='->', color='red', lw=1.2))
ax_p.annotate(f"Post-policy peak: {airchina_stock.loc[mx_post,'close']:.2f} HKD",
              xy=(airchina_stock.loc[mx_post,'date'], airchina_stock.loc[mx_post,'close']),
              xytext=(-80, -20), textcoords='offset points', fontsize=9,
              color='green', fontweight='bold',
              arrowprops=dict(arrowstyle='->', color='green', lw=1.2))

gain = (airchina_stock.loc[mx_post,'close'] - airchina_stock.loc[mi,'close']) / airchina_stock.loc[mi,'close'] * 100
ax_p.text(0.02, 0.05, f'Trough-to-Peak Gain: +{gain:.1f}%',
          transform=ax_p.transAxes, fontsize=11, color='green',
          bbox=dict(facecolor='lightyellow', alpha=0.8))

for ds, label in urumqi_events.items():
    dt = pd.Timestamp(ds)
    ax_p.axvline(x=dt, color='gray', linestyle='--', alpha=0.5)
    ax_p.annotate(label, xy=(dt, ax_p.get_ylim()[1] * 0.93),
                  fontsize=7.5, ha='center', va='top',
                  bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

ax_p.set_ylabel('Closing Price (HKD)')
ax_p.legend(loc='upper left')

colors_v = ['#E74C3C' if r < 0 else '#2ECC71'
            for r in airchina_stock['daily_return'].fillna(0)]
ax_v.bar(airchina_stock['date'], airchina_stock['volume'] / 1e6,
         width=1, color=colors_v, alpha=0.7)

# Annotate the massive Dec 7 volume
dec7_idx = airchina_stock[airchina_stock['date'] == pd.Timestamp('2022-12-07')].index
if len(dec7_idx) > 0:
    dec7_vol = airchina_stock.loc[dec7_idx[0], 'volume'] / 1e6
    ax_v.annotate(f'Dec 7\n{dec7_vol:.0f}M\n(policy reversal)',
                  xy=(pd.Timestamp('2022-12-07'), dec7_vol),
                  xytext=(20, -5), textcoords='offset points', fontsize=8,
                  color='#E74C3C', fontweight='bold',
                  arrowprops=dict(arrowstyle='->', color='#E74C3C'))

ax_v.set_ylabel('Volume (M)')
ax_v.set_xlabel('Date')
ax_p.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax_p.xaxis.set_major_locator(mdates.MonthLocator())
ax_p.set_title('Air China (0753.HK) Stock Price & Volume\n'
               'Urumqi Fire Crisis & Zero COVID Policy Reversal (Oct 2022 – Feb 2023)',
               fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/C3_Fig2_AirChina_Price_Volume.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Figure 3: Haidilao & Trip.com — Reopening Stocks Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10),
                         gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.05, 'wspace': 0.3})
(ax_hp, ax_hv), (ax_tp, ax_tv) = axes[:, 0], axes[:, 1]

for ax in [ax_hp, ax_hv, ax_tp, ax_tv]:
    ax.axvspan(*protest_period, alpha=0.10, color='red')
    ax.axvspan(reopening_start, pd.Timestamp('2023-02-28'), alpha=0.06, color='green')

# --- Haidilao ---
ax_hp.plot(haidilao_stock['date'], haidilao_stock['close'],
           color='#E67E22', linewidth=1.8, label='Haidilao (HKD)')
hd_mi = haidilao_stock['close'].idxmin()
hd_mx = haidilao_stock[haidilao_stock['date'] >= reopening_start]['close'].idxmax()
hd_gain = (haidilao_stock.loc[hd_mx,'close'] - haidilao_stock.loc[hd_mi,'close']) / haidilao_stock.loc[hd_mi,'close'] * 100
ax_hp.text(0.03, 0.05, f'Trough-to-Peak: +{hd_gain:.1f}%',
           transform=ax_hp.transAxes, fontsize=10, color='green',
           bbox=dict(facecolor='lightyellow', alpha=0.8))
for ds, label in urumqi_events.items():
    ax_hp.axvline(x=pd.Timestamp(ds), color='gray', linestyle='--', alpha=0.4)
ax_hp.set_ylabel('Price (HKD)')
ax_hp.set_title('Haidilao (6862.HK)', fontweight='bold')
ax_hp.legend(loc='upper left', fontsize=9)
ax_hp.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

colors_hd = ['#E74C3C' if r < 0 else '#2ECC71'
             for r in haidilao_stock['daily_return'].fillna(0)]
ax_hv.bar(haidilao_stock['date'], haidilao_stock['volume'] / 1e6,
          width=1, color=colors_hd, alpha=0.7)
ax_hv.set_ylabel('Volume (M)')
ax_hv.set_xlabel('Date')
ax_hv.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

# --- Trip.com ---
ax_tp.plot(tripcom_stock['date'], tripcom_stock['close'],
           color='#9B59B6', linewidth=1.8, label='Trip.com (USD)')
tc_mi = tripcom_stock['close'].idxmin()
tc_mx = tripcom_stock[tripcom_stock['date'] >= reopening_start]['close'].idxmax()
tc_gain = (tripcom_stock.loc[tc_mx,'close'] - tripcom_stock.loc[tc_mi,'close']) / tripcom_stock.loc[tc_mi,'close'] * 100
ax_tp.text(0.03, 0.05, f'Trough-to-Peak: +{tc_gain:.1f}%',
           transform=ax_tp.transAxes, fontsize=10, color='green',
           bbox=dict(facecolor='lightyellow', alpha=0.8))
for ds, label in urumqi_events.items():
    ax_tp.axvline(x=pd.Timestamp(ds), color='gray', linestyle='--', alpha=0.4)
ax_tp.set_ylabel('Price (USD)')
ax_tp.set_title('Trip.com (TCOM)', fontweight='bold')
ax_tp.legend(loc='upper left', fontsize=9)
ax_tp.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

colors_tc = ['#E74C3C' if r < 0 else '#2ECC71'
             for r in tripcom_stock['daily_return'].fillna(0)]
ax_tv.bar(tripcom_stock['date'], tripcom_stock['volume'] / 1e6,
          width=1, color=colors_tc, alpha=0.7)
ax_tv.set_ylabel('Volume (M)')
ax_tv.set_xlabel('Date')
ax_tv.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.suptitle('Reopening-Sensitive Stocks: Haidilao & Trip.com\n'
             'Red = Protest Period | Green = Reopening Rally',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/C3_Fig3_Haidilao_TripCom_Price_Volume.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.4 Figure 4: Related Search Queries — Urumqi Fire

In [ ]:
uru_top  = pd.read_csv('data/urumqi_top_queries.csv')
uru_rise = pd.read_csv('data/urumqi_rising_queries.csv')
uru_top.columns = uru_rise.columns = ['query', 'search_interest', 'change_pct']

fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 6))

td = uru_top.head(10).sort_values('search_interest')
a1.barh(range(len(td)), td['search_interest'], color='#E74C3C', alpha=0.8)
a1.set_yticks(range(len(td)))
a1.set_yticklabels(td['query'], fontsize=9)
a1.set_xlabel('Search Interest')
a1.set_title('Top Related Queries\n"Urumqi fire" (Oct 2022 – Feb 2023)', fontweight='bold')
for i, (_, r) in enumerate(td.iterrows()):
    a1.text(r['search_interest'] + 1, i, str(r['change_pct']), fontsize=8, va='center')

rd = uru_rise.head(10).sort_values('search_interest')
a2.barh(range(len(rd)), rd['search_interest'],
        color=['#E74C3C' if str(c) == 'Breakout' else '#F39C12' for c in rd['change_pct']],
        alpha=0.8)
a2.set_yticks(range(len(rd)))
a2.set_yticklabels(rd['query'], fontsize=9)
a2.set_xlabel('Search Interest')
a2.set_title('Rising Related Queries\n"Urumqi fire" (Oct 2022 – Feb 2023)', fontweight='bold')
for i, (_, r) in enumerate(rd.iterrows()):
    a2.text(r['search_interest'] + 0.5, i, str(r['change_pct']), fontsize=8, va='center',
            color='#E74C3C' if str(r['change_pct']) == 'Breakout' else '#333')

n_bo = (uru_rise['change_pct'] == 'Breakout').sum()
plt.suptitle(f'Google Trends: Related Search Queries for "Urumqi fire" ({n_bo} Breakout queries)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/C3_Fig4_Urumqi_Queries.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.5 Figure 5: Related Search Queries — China Protests

In [ ]:
pr_top  = pd.read_csv('data/protests_top_queries.csv')
pr_rise = pd.read_csv('data/protests_rising_queries.csv')
pr_top.columns = pr_rise.columns = ['query', 'search_interest', 'change_pct']

fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 7))

td = pr_top.head(12).sort_values('search_interest')
# Highlight queries that mention COVID/lockdown (the political framing)
bar_colors = ['#E74C3C' if any(k in str(q).lower() for k in ['covid', 'zero', 'lockdown'])
              else '#3498DB' for q in td['query']]
a1.barh(range(len(td)), td['search_interest'], color=bar_colors, alpha=0.8)
a1.set_yticks(range(len(td)))
a1.set_yticklabels(td['query'], fontsize=9)
a1.set_xlabel('Search Interest')
a1.set_title('Top Related Queries\n"China protests"  (Red = COVID/lockdown framing)', fontweight='bold')
for i, (_, r) in enumerate(td.iterrows()):
    a1.text(r['search_interest'] + 1, i, str(r['change_pct']), fontsize=8, va='center')

rd = pr_rise.head(12).sort_values('search_interest')
a2.barh(range(len(rd)), rd['search_interest'],
        color=['#E74C3C' if str(c) == 'Breakout' else '#F39C12' for c in rd['change_pct']],
        alpha=0.8)
a2.set_yticks(range(len(rd)))
a2.set_yticklabels(rd['query'], fontsize=9)
a2.set_xlabel('Search Interest')
a2.set_title('Rising Related Queries\n"China protests"', fontweight='bold')
for i, (_, r) in enumerate(rd.iterrows()):
    a2.text(r['search_interest'] + 0.5, i, str(r['change_pct']), fontsize=8, va='center',
            color='#E74C3C' if str(r['change_pct']) == 'Breakout' else '#333')

plt.suptitle('Google Trends: Related Queries for "China protests"\n'
             'Dominated by COVID/Zero COVID framing — political nature evident',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/C3_Fig5_Protests_Queries.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Statistical Analysis

### 6.1 Descriptive Statistics

Three periods defined:
- **Pre-crisis**: before Nov 24 (Urumqi fire)
- **Crisis/Protest window**: Nov 24 – Dec 7 (fire to policy reversal)
- **Post-policy reversal**: after Dec 7

In [ ]:
def reopening_summary(stock_df, name, cs, ce):
    pre  = stock_df[stock_df['date'] <  cs]
    dur  = stock_df[(stock_df['date'] >= cs) & (stock_df['date'] <= ce)]
    post = stock_df[stock_df['date'] >  ce]

    trough = stock_df['close'].min()
    peak_post = post['close'].max() if len(post) > 0 else float('nan')
    gain = (peak_post - trough) / trough * 100 if not np.isnan(peak_post) else float('nan')

    print(f"\n{'='*65}")
    print(f"{name}")
    print(f"{'='*65}")
    print(f"Pre-crisis avg:  {pre['close'].mean():.2f}  |  "
          f"Protest window avg: {dur['close'].mean():.2f}  |  "
          f"Post-reversal avg: {post['close'].mean():.2f}")
    print(f"Trough: {trough:.2f}  |  Post-reversal peak: {peak_post:.2f}  |  Gain: +{gain:.1f}%")
    if len(pre) > 0 and pre['volume'].mean() > 0:
        vol_ratio = dur['volume'].mean() / pre['volume'].mean()
        print(f"Volume: pre {pre['volume'].mean()/1e6:.1f}M -> protest window "
              f"{dur['volume'].mean()/1e6:.1f}M ({(vol_ratio-1)*100:.0f}% change)")

reopening_summary(airchina_stock,  'Air China (0753.HK)',  *protest_period)
reopening_summary(haidilao_stock,  'Haidilao (6862.HK)',   *protest_period)
reopening_summary(tripcom_stock,   'Trip.com (TCOM)',      *protest_period)

### 6.2 Correlation Analysis

**Hypothesis**: As protest search interest rises, reopening-stock prices rise in anticipation of policy reversal (positive correlation — opposite to Case 1).

In [ ]:
def correlation_analysis(mdf, tcol, scol='avg_close', name='Case'):
    v = mdf[[tcol, scol]].dropna()
    pr, pp = stats.pearsonr(v[tcol], v[scol])
    sr, sp = stats.spearmanr(v[tcol], v[scol])
    print(f"\n{'='*55}")
    print(f"Correlation: {name}")
    print(f"Pearson  r={pr:.4f} (p={pp:.4f}){'*' if pp < 0.05 else ''}  |  "
          f"Spearman rho={sr:.4f} (p={sp:.4f}){'*' if sp < 0.05 else ''}")

    ml = min(5, len(v) // 3)
    lags = []
    for lag in range(-ml, ml + 1):
        x = v[tcol].iloc[max(0, -lag):len(v) - max(0, lag)].values
        y = v[scol].iloc[max(0,  lag):len(v) - max(0,-lag)].values
        if len(x) > 3:
            r, p = stats.pearsonr(x, y)
            lags.append({'lag': lag, 'r': r, 'p': p})
    ldf = pd.DataFrame(lags)
    best = ldf.loc[ldf['r'].abs().idxmax()]
    print(f"Best lag: {int(best['lag'])} weeks, r={best['r']:.4f} (p={best['p']:.4f})")

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(ldf['lag'], ldf['r'],
           color=['#E74C3C' if p < 0.05 else '#BDC3C7' for p in ldf['p']])
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.set_xlabel('Lag (weeks, + = trend leads stock)')
    ax.set_ylabel('Pearson r')
    ax.set_title(f'Cross-Correlation: {name}\n(Red = p < 0.05)')
    plt.tight_layout()
    plt.savefig(f'figures/C3_XCorr_{name.replace(" ","_")}.png', dpi=150)
    plt.show()
    return ldf

tcol = 'China protests'
lag_ac = correlation_analysis(urumqi_merged_ac, tcol, name='Urumqi - Air China')
lag_hd = correlation_analysis(urumqi_merged_hd, tcol, name='Urumqi - Haidilao')
lag_tc = correlation_analysis(urumqi_merged_tc, tcol, name='Urumqi - Trip.com')

### 6.3 Granger Causality Test

- **H0**: "China protests" search interest does NOT Granger-cause stock price  
- **H1**: "China protests" search interest DOES Granger-cause stock price  
- Reject H0 if p < 0.05  

Expected direction: **positive** (more protests → market anticipates reopening → prices rise)

In [ ]:
def check_stat(s, name):
    r = adfuller(s.dropna(), autolag='AIC')
    ok = r[1] < 0.05
    print(f"  ADF {name}: stat={r[0]:.3f}, p={r[1]:.4f} -> "
          f"{'Stationary' if ok else 'Non-stationary (diff)'}")
    return ok

def granger_test(mdf, tcol, scol='avg_close', ml=3, name='Case'):
    print(f"\n{'='*65}")
    print(f"Granger Causality: {name} (n={len(mdf.dropna(subset=[tcol, scol]))})")
    print(f"{'='*65}")
    df = mdf[[tcol, scol]].dropna().copy()
    tu, su = tcol, scol
    if not check_stat(df[tcol], tcol):
        df[tcol + '_d'] = df[tcol].diff()
        tu = tcol + '_d'
    if not check_stat(df[scol], scol):
        df[scol + '_d'] = df[scol].diff()
        su = scol + '_d'
    df = df.dropna()
    if len(df) < ml + 5:
        ml = max(1, len(df) // 4)
        print(f"  Reduced max_lag to {ml}")

    print(f"\nForward: {tcol} -> {scol}")
    try:
        rf = grangercausalitytests(df[[su, tu]].values, maxlag=ml, verbose=False)
        for lag in range(1, ml + 1):
            f, p = rf[lag][0]['ssr_ftest'][:2]
            print(f"  Lag {lag}: F={f:.3f}, p={p:.4f} {'YES *' if p < 0.05 else 'no'}")
    except Exception as e:
        print(f"  Error: {e}")

    print(f"\nReverse: {scol} -> {tcol}")
    try:
        rr = grangercausalitytests(df[[tu, su]].values, maxlag=ml, verbose=False)
        for lag in range(1, ml + 1):
            f, p = rr[lag][0]['ssr_ftest'][:2]
            print(f"  Lag {lag}: F={f:.3f}, p={p:.4f} {'YES *' if p < 0.05 else 'no'}")
    except Exception as e:
        print(f"  Error: {e}")

tcol = 'China protests'
granger_test(urumqi_merged_ac, tcol, ml=3, name='Urumqi - Air China')
granger_test(urumqi_merged_hd, tcol, ml=3, name='Urumqi - Haidilao')
granger_test(urumqi_merged_tc, tcol, ml=3, name='Urumqi - Trip.com')

### 6.4 Event Study: Cumulative Abnormal Returns around Dec 7 Policy Reversal

In [ ]:
def event_study(sdf, edate, ename, wb=10, wa=20):
    edt = pd.Timestamp(edate)
    df  = sdf.copy()
    candidates = df[df['date'] >= edt].index
    if len(candidates) == 0:
        print(f"No data from {edate} onwards for {ename}")
        return None
    ei = candidates[0]
    nr = df.loc[max(0, ei - wb - 30):ei - wb, 'daily_return'].mean()
    w  = df.loc[max(0, ei - wb):min(len(df) - 1, ei + wa)].copy()
    w['ar']  = w['daily_return'] - nr
    w['CAR'] = w['ar'].cumsum()
    w['day'] = range(-min(wb, ei - w.index[0]), len(w) - min(wb, ei - w.index[0]))

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(w['day'], w['ar'] * 100,
           color=['#E74C3C' if x < 0 else '#2ECC71' for x in w['ar']],
           alpha=0.6, label='Daily AR%')
    ax.plot(w['day'], w['CAR'] * 100, 'b-o',
            linewidth=2, markersize=4, label='CAR%')
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2,
               label=f'Event: {ename}')
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.set_xlabel('Trading Days Relative to Event')
    ax.set_ylabel('Return (%)')
    ax.set_title(f'Event Study: {ename}\nCumulative Abnormal Returns', fontsize=14)
    ax.legend()
    plt.tight_layout()
    safe = ename.replace(' ', '_').replace('/', '-')
    plt.savefig(f'figures/C3_ES_{safe}.png', dpi=150)
    plt.show()
    print(f"CAR at day +{wa}: {w['CAR'].iloc[-1]*100:.2f}%")
    return w

# Event: Dec 7, 2022 — Zero COVID policy reversal (新十条)
es_ac = event_study(airchina_stock, '2022-12-07', 'Zero COVID Ended - Air China',  wb=10, wa=20)
es_hd = event_study(haidilao_stock, '2022-12-07', 'Zero COVID Ended - Haidilao',   wb=10, wa=20)
es_tc = event_study(tripcom_stock,  '2022-12-07', 'Zero COVID Ended - Trip.com',   wb=10, wa=20)

## 7. Cross-Stock Comparison

In [ ]:
# Normalized trajectories from protest start (Nov 24, 2022 = 100)
fig, ax = plt.subplots(figsize=(14, 7))

ax.axvspan(*protest_period, alpha=0.10, color='red',   label='Protest Window')
ax.axvspan(reopening_start, pd.Timestamp('2023-02-28'),
           alpha=0.06, color='green', label='Reopening Rally')

baseline = pd.Timestamp('2022-11-24')
stock_configs = [
    (airchina_stock,  'Air China (0753.HK, HKD)', '#F39C12'),
    (haidilao_stock,  'Haidilao (6862.HK, HKD)',  '#E67E22'),
    (tripcom_stock,   'Trip.com (TCOM, USD)',      '#9B59B6'),
]

for sdf, nm, c in stock_configs:
    s = sdf.copy()
    p0_row = s[s['date'] >= baseline]
    if len(p0_row) == 0:
        continue
    p0 = p0_row.iloc[0]['close']
    s['norm'] = s['close'] / p0 * 100
    s['d']    = (s['date'] - baseline).dt.days
    ax.plot(s['d'], s['norm'], linewidth=2.2, label=nm, color=c)

ax.axhline(y=100, color='gray', linestyle='--', alpha=0.5, label='Baseline (Nov 24)')
ax.axvline(x=0,   color='darkred',  linestyle='--', alpha=0.7)
ax.axvline(x=13,  color='darkgreen', linestyle='--', alpha=0.7)
ax.text(0.5,  101, 'Fire', fontsize=9, color='darkred')
ax.text(13.5, 101, 'Policy\nReversal', fontsize=9, color='darkgreen')

ax.set_xlabel('Days from Urumqi Fire (Nov 24, 2022)')
ax.set_ylabel('Normalized Price (Nov 24 = 100)')
ax.legend(loc='upper left', fontsize=10)
plt.title('Case 3: All Three Reopening Stocks — Normalized Price Trajectory\n'
          'From Urumqi Fire (Nov 24) through Policy Reversal (Dec 7) and Rally',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/C3_Fig6_CrossStock_Normalized.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary comparison table
def stock_stats(sdf, name):
    pre_end = protest_period[0]
    post_start = protest_period[1]
    trough = sdf['close'].min()
    peak_post = sdf[sdf['date'] >= post_start]['close'].max()
    gain = (peak_post - trough) / trough * 100
    pre_vol  = sdf[sdf['date'] <  pre_end]['volume'].mean() / 1e6
    prot_vol = sdf[(sdf['date'] >= pre_end) & (sdf['date'] <= post_start)]['volume'].mean() / 1e6
    return {
        'Stock': name,
        'Trough (pre-reversal)': f"{trough:.2f}",
        'Post-reversal peak': f"{peak_post:.2f}",
        'Trough-to-peak gain': f"+{gain:.1f}%",
        'Pre vol (M/day)': f"{pre_vol:.1f}",
        'Protest vol (M/day)': f"{prot_vol:.1f}",
    }

rows = [
    stock_stats(airchina_stock, 'Air China 0753.HK (HKD)'),
    stock_stats(haidilao_stock, 'Haidilao 6862.HK (HKD)'),
    stock_stats(tripcom_stock,  'Trip.com TCOM (USD)'),
]
comp = pd.DataFrame(rows)
print(comp.to_string(index=False))

## 8. Key Findings

**1. Three-stage causal chain confirmed** — unlike Cases 1 and 2 (trend → market), Case 3 documents: *Social Media Pressure → Government Policy Reversal → Market Recovery*.

**2. Positive trend–stock correlation** — search interest for "China protests" correlates *positively* with reopening stock prices, the reverse of Case 1 (Nongfu boycott). The market was pricing in the probability of policy change, not reacting to damage.

**3. Speed of policy response** — the Zero COVID reversal occurred just **13 days** after protest search interest peaked at 100 on the week of Nov 27. This is the fastest policy response across all four cases.

**4. Market pre-priced the reversal** — Air China and Haidilao began rising in the protest window (before Dec 7), suggesting markets anticipated the policy change from online signals alone.

**5. Dec 7 volume spike** — Air China traded **128M shares** on Dec 7 (the policy announcement day), the highest single-day volume in the dataset, confirming the market's sharp reaction.

**6. Censorship paradox** — despite heavy platform censorship, the intensity of the signal (protests trending globally, "Urumqi fire" all-Breakout queries) was sufficient to force government action. This illustrates the limits of information control under extreme public pressure.

### Limitations
- Google Trends captures *global* searches; domestic (Weibo/Douyin) intensity was likely far higher but unobservable via public API
- Only 23 weekly observations — Granger results should be interpreted cautiously
- The three stocks (HKD, HKD, USD) are on different currency bases; cross-stock comparison uses normalized prices
- Policy reversal may have been driven by factors beyond social media (economic costs of Zero COVID) — Granger causality ≠ true causation